# Génération de représentations 3D augmentées

Ce notebook convertit les SMILES en géométries 3D et génère plusieurs variantes augmentées par molécule.

Les résultats seront sauvegardés dans `data/augmented_3d_representations.pt`.

## Configuration

Le CSV contient `nonStereoSMILES`, `descriptors`, puis une colonne binaire par label olfactif.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from rdkit import Chem
from rdkit.Chem import AllChem
from tqdm.auto import tqdm

# Configuration
CSV_PATH = Path("../../data/smiles_odor_dataset.csv")
OUTPUT_PATH = Path("../../data/conformers_3d_dataset_100_000.pt")
MAX_CONFS_PER_MOL = 15  # Génère jusqu'à 15 conformères 3D variés par molécule
RMSD_THRESHOLD = 0.25   # Écarte les conformères 3D trop identiques (en Angstroms)
RANDOM_SEED = 42

# 5000 smiles x 15(MAX_CONFS_PER_MOL) + 1/3(mol non-opti) = 100_000 (x20)

## Conversion chimique et augmentation

`MolFromSmiles` transforme le texte SMILES en graphe moléculaire RDKit. `AddHs` rend explicites les hydrogènes, afin qu'ils reçoivent eux aussi des coordonnées.

ETKDG génère des conformations 3D compatibles avec la topologie chimique. MMFF ajuste ensuite leur géométrie pour réduire leur énergie.

In [2]:
def generate_diverse_conformers(
    smiles: str,
    max_confs: int = 15,
    rmsd_thresh: float = 0.25,
    seed: int = 42,
) -> dict[str, object]:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"SMILES invalide : {smiles!r}")

    # Ajout des hydrogènes explicites (requis par Uni-Mol)
    mol = Chem.AddHs(mol)

    # Configuration ETKDGv3 avec élagage par diversité (RMSD)
    params = AllChem.ETKDGv3()
    params.randomSeed = seed
    params.pruneRmsThresh = rmsd_thresh  # enlève les conformomères trop similaire 
    params.useSmallRingTorsions = True 

    conf_ids = list(AllChem.EmbedMultipleConfs(mol, numConfs=max_confs, params=params))

    # Fallback si la molécule est très rigide ou encombrée
    if not conf_ids:
        params.pruneRmsThresh = -1.0
        params.useRandomCoords = True
        conf_ids = list(AllChem.EmbedMultipleConfs(
            mol, 
            numConfs=5 if max_confs > 5 else max_confs,
            params=params
            ))
        if not conf_ids:
            raise ValueError(f"Impossible de générer la 3D pour {smiles!r}")

    coords_list = []
    
    # 1. On conserve quelques géométries ETKDG brutes (avant relaxation MMFF)
    for cid in conf_ids[: max(1, len(conf_ids) // 3)]:
        pos = np.asarray(mol.GetConformer(cid).GetPositions(), dtype=np.float32)
        coords_list.append(torch.from_numpy(pos))

    # 2. Optimisation énergétique MMFF94 sur tous les conformères
    try:
        AllChem.MMFFOptimizeMoleculeConfs(mol, maxIters=200)
    except (RuntimeError, ValueError):
        pass

    for cid in conf_ids:
        pos = np.asarray(mol.GetConformer(cid).GetPositions(), dtype=np.float32)
        coords_list.append(torch.from_numpy(pos))

    # Extraction des informations atomiques (communes à tous les conformères)
    atom_symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
    atomic_numbers = torch.tensor(
        [atom.GetAtomicNum() for atom in mol.GetAtoms()], dtype=torch.int64
    )

    return {
        "smiles": smiles,
        "atom_symbols": atom_symbols,        # Utile pour le vocabulaire Uni-Mol ('C', 'O', 'H'...)
        "atomic_numbers": atomic_numbers,    # Shape: [N_atoms]
        "conformers": coords_list,           # Liste de tenseurs [N_atoms, 3]
    }

## Génération et sauvegarde

Les molécules peuvent avoir des nombres d'atomes différents. Les représentations sont donc conservées dans une liste plutôt que dans un tenseur dense unique.

In [ ]:
df = pd.read_csv(CSV_PATH)
smiles_list = df["nonStereoSMILES"].astype(str).tolist()
label_columns = [c for c in df.columns if c not in {"nonStereoSMILES", "descriptors"}]
labels_matrix = df[label_columns].to_numpy(dtype=np.float32)

dataset_items = []
failed_molecules = []

for idx, (smiles, labels) in enumerate(
    tqdm(zip(smiles_list, labels_matrix, strict=True), total=len(smiles_list))
):
    try:
        item = generate_diverse_conformers(
            smiles=smiles,
            max_confs=MAX_CONFS_PER_MOL,
            rmsd_thresh=RMSD_THRESHOLD,
            seed=RANDOM_SEED + idx,
        )
        item["source_index"] = idx
        item["labels"] = torch.from_numpy(labels.copy())
        dataset_items.append(item)
    except Exception as e:
        failed_molecules.append({"source_index": idx, "smiles": smiles, "error": str(e)})

payload = {
    "molecules": dataset_items,  # 1 entrée par molécule contenant K conformères
    "label_names": label_columns,
    "failed_molecules": failed_molecules,
}

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(payload, OUTPUT_PATH)
print(f"Terminé : {len(dataset_items)} molécules sauvegardées ({len(failed_molecules)} échecs).")

  0%|          | 0/4983 [00:00<?, ?it/s]

[14:46:17] 

****
Invariant Violation
bad direction in linearSearch
Violation occurred on line 234 in file /Users/runner/work/rdkit-pypi/rdkit-pypi/build/temp.macosx-11.0-arm64-cpython-312/rdkit/Code/Numerics/Optimizer/BFGSOpt.h
Failed Expression: status >= 0
****

[14:47:31] 

****
Invariant Violation
bad direction in linearSearch
Violation occurred on line 234 in file /Users/runner/work/rdkit-pypi/rdkit-pypi/build/temp.macosx-11.0-arm64-cpython-312/rdkit/Code/Numerics/Optimizer/BFGSOpt.h
Failed Expression: status >= 0
****

[14:49:39] 

****
Invariant Violation
bad direction in linearSearch
Violation occurred on line 234 in file /Users/runner/work/rdkit-pypi/rdkit-pypi/build/temp.macosx-11.0-arm64-cpython-312/rdkit/Code/Numerics/Optimizer/BFGSOpt.h
Failed Expression: status >= 0
****

